In [2]:
! pip install unsloth trl datasets transformers bitsandbytes accelerate vllm
! pip install -U transformers trl huggingface_hub mergekit
! pip install flashinfer-cubin==0.6.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 MB 14.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 MB 3.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 97.8 MB/s eta 0:00:00:

  Using cached trl-1.0.0-py3-none-any.whl.metadata (11 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7

# For Instrucct models

### Load model

In [ ]:
# 1. UNSLOTH MUST BE IMPORTED FIRST
from unsloth import FastLanguageModel

# 2. Then import everything else
import os
import re
import sys
import json
import tempfile
import subprocess
from datasets import load_dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False, # <--- SET THIS TO FALSE
)



max_seq_length = 2048

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)


#### Prepare Dataset

In [ ]:

# ==========================================
# 2. DATASET PREPARATION
# ==========================================
def format_dataset(row):
    # Parse the codeforces examples JSON string into inputs and outputs
    try:
        examples = json.loads(row['examples'])
        input_tests = [str(ex['input']) for ex in examples]
        output_tests = [str(ex['output']) for ex in examples]
    except Exception:
        input_tests = []
        output_tests = []

    question_level = row['id'].split('/')[-1] if '/' in row['id'] else ""

    return {
        "prompt": [{"role": "user", "content": row['prompt']}], # TRL expects ChatML format
        "input_tests": input_tests,
        "output_tests": output_tests,
        "question_level": question_level
    }

print("Loading and filtering dataset...")
# Load dataset and filter for 'A' questions
dataset = load_dataset("open-r1/codeforces-cots", split="train[:100]")

# Filter for question A and apply formatting
filtered_dataset = dataset.filter(lambda x: x['id'].endswith('A'))
processed_dataset = filtered_dataset.map(format_dataset, num_proc=4)

# Take subset for training
train_dataset = processed_dataset.select(range(min(1000, len(processed_dataset))))


#### Define Reward functions

In [1]:
# ==========================================
# 3. REWARD FUNCTIONS (DeepSeek Strategy)
# ==========================================
def format_reward_func(completions, **kwargs):
    """Rewards strict adherence to <think> and markdown code block formatting."""
    rewards = []
    for c in completions:
        # Completions come as a list of dicts or strings depending on TRL version
        content = c[0]['content'] if isinstance(c, list) else c

        has_think = bool(re.search(r"<think>.*?</think>", content, re.DOTALL))
        has_code = bool(re.search(r"```(python|cpp)\n.*?```", content, re.DOTALL))
        rewards.append(0.2 if (has_think and has_code) else 0.0)
    return rewards

def extract_code(completion):
    match = re.search(r"```(python|cpp)\n(.*?)```", completion, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

def run_code(code, lang, inp):
    with tempfile.TemporaryDirectory() as tmp:
        if lang == "cpp":
            cpp = os.path.join(tmp, "sol.cpp")
            exe = os.path.join(tmp, "sol")
            with open(cpp, "w") as f: f.write(code)

            if subprocess.run(["g++", "-O2", cpp, "-o", exe], capture_output=True).returncode != 0:
                return None, "ce" # Compilation Error
            cmd = [exe]
        else:
            py = os.path.join(tmp, "sol.py")
            with open(py, "w") as f: f.write(code)
            cmd = [sys.executable, py]

        try:
            res = subprocess.run(cmd, input=inp, text=True, capture_output=True, timeout=2.0)
            if res.returncode != 0:
                return None, "re" # Runtime Error
            return res.stdout.strip(), "ok"
        except subprocess.TimeoutExpired:
            return None, "tle" # Time Limit Exceeded
        except Exception as e:
            return None, f"error: {str(e)}"

def correctness_reward_func(completions, input_tests, output_tests, **kwargs):
    """Executes the code and scores based on percentage of passed test cases."""
    rewards = []
    for comp, ins, outs in zip(completions, input_tests, output_tests):
        content = comp[0]['content'] if isinstance(comp, list) else comp
        lang, code = extract_code(content)

        if not code or not ins:
            rewards.append(-0.5)
            continue

        score = 0
        total = len(ins)

        for i, o in zip(ins, outs):
            out, status = run_code(code, lang, i)
            if status == "ok" and out == o.strip():
                score += 1

        # Proportional reward based on test cases passed
        rewards.append(score / total if total > 0 else 0.0)

    return rewards


ModuleNotFoundError: No module named 'unsloth'

#### Training

In [ ]:

# ==========================================
# 4. TRAINING & LOGGING CALLBACK
# ==========================================
class EpochSaveCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        # Save to the exact same directory, overwriting previous epoch
        save_path = os.path.join(args.output_dir, "latest_epoch_weights")
        print(f"\n[Epoch {state.epoch}] Saving overwritten checkpoint to {save_path}")
        kwargs["model"].save_pretrained(save_path)
        kwargs["tokenizer"].save_pretrained(save_path)

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, 
    num_generations = 4,
    # max_prompt_length = 1024,
    # max_completion_length = 1024,
    num_train_epochs = 1,
    save_strategy="no",
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "outputs",
    importance_sampling_level = "sequence",
    loss_type = "dr_grpo",
    save_strategy="steps",   # or "epoch"
    save_steps=2,          # save every 1 steps
    save_total_limit=2,      # keep last 3 checkpoints
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
    callbacks=[EpochSaveCallback()]
)

print("Starting GRPO Training...")
trainer.train()



#### Post Training Verification


In [ ]:

# ==========================================
# 5. POST-TRAINING VERIFICATION
# ==========================================
print("\n--- Running Manual Verification ---")
# Pick a question outside our train set (e.g., from the end of the dataset)
val_sample = processed_dataset[-1]

# Format prompt for generation
messages = val_sample["prompt"]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

# Generate Code
print(f"Generating solution for Problem: {val_sample['id']}")
outputs = model.generate(
    inputs,
    max_new_tokens=1024,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("\n--- Generated Output ---")
print(generated_text)
print("------------------------")

# Run through custom reward functions to check outputs manually
format_score = format_reward_func([generated_text])[0]
correctness_score = correctness_reward_func(
    [generated_text],
    [val_sample["input_tests"]],
    [val_sample["output_tests"]]
)[0]

print("\n--- Metrics ---")
print(f"Format Reward  : {format_score}")
print(f"Correctness    : {correctness_score} (Percentage of Test Cases Passed)")

# for non instruct models

In [ ]:
# 1. UNSLOTH MUST BE IMPORTED FIRST
from unsloth import FastLanguageModel

# 2. Then import everything else
import os
import re
import sys
import json
import tempfile
import subprocess
from datasets import load_dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer

max_seq_length = 2048

# Use a BASE model instead of an Instruct model
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/Qwen2.5-0.5B", 
    model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit", 
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False, # <--- SET THIS TO FALSE
)

# Ensure pad token is set for base models (often required for batched generation)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

# ==========================================
# 2. DATASET PREPARATION
# ==========================================
def format_dataset(row):
    try:
        examples = json.loads(row['examples'])
        input_tests = [str(ex['input']) for ex in examples]
        output_tests = [str(ex['output']) for ex in examples]
    except Exception:
        input_tests = []
        output_tests = []
    
    question_level = row['id'].split('/')[-1] if '/' in row['id'] else ""
    
    # BASE MODEL CHANGE: Use a raw string with strict formatting instead of chat dicts.
    # We explicitly guide the base model on how to format its output.
    raw_prompt = (
        f"You are an expert competitive programmer. Solve the following problem.\n"
        f"Put your reasoning inside <think> </think> tags, and your code inside a markdown block.\n\n"
        f"Problem:\n{row['prompt']}\n\n"
        f"Solution:\n<think>\n"
    )
    
    return {
        "prompt": raw_prompt,
        "input_tests": input_tests,
        "output_tests": output_tests,
        "question_level": question_level
    }

print("Loading and filtering dataset...")
# Load dataset and filter for 'A' questions
dataset = load_dataset("open-r1/codeforces-cots", split="train[:1000]")

filtered_dataset = dataset.filter(lambda x: x['id'].endswith('A'))
processed_dataset = filtered_dataset.map(format_dataset, num_proc=4)

train_dataset = processed_dataset.select(range(min(1000, len(processed_dataset))))

# ==========================================
# 3. REWARD FUNCTIONS
# ==========================================
def format_reward_func(completions, **kwargs):
    rewards = []
    for c in completions:
        # TRL passes string completions for base string prompts
        content = c[0]['content'] if isinstance(c, list) else c
        
        has_think = bool(re.search(r"<think>.*?</think>", content, re.DOTALL))
        has_code = bool(re.search(r"```(python|cpp)\n.*?```", content, re.DOTALL))
        rewards.append(0.2 if (has_think and has_code) else 0.0)
    return rewards

def extract_code(completion):
    match = re.search(r"```(python|cpp)\n(.*?)```", completion, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

def run_code(code, lang, inp):
    with tempfile.TemporaryDirectory() as tmp:
        if lang == "cpp":
            cpp = os.path.join(tmp, "sol.cpp")
            exe = os.path.join(tmp, "sol")
            with open(cpp, "w") as f: f.write(code)

            if subprocess.run(["g++", "-O2", cpp, "-o", exe], capture_output=True).returncode != 0:
                return None, "ce"
            cmd = [exe]
        else:
            py = os.path.join(tmp, "sol.py")
            with open(py, "w") as f: f.write(code)
            cmd = [sys.executable, py]

        try:
            res = subprocess.run(cmd, input=inp, text=True, capture_output=True, timeout=2.0)
            if res.returncode != 0:
                return None, "re"
            return res.stdout.strip(), "ok"
        except subprocess.TimeoutExpired:
            return None, "tle"
        except Exception as e:
            return None, f"error: {str(e)}"

def correctness_reward_func(completions, input_tests, output_tests, **kwargs):
    rewards = []
    for comp, ins, outs in zip(completions, input_tests, output_tests):
        content = comp[0]['content'] if isinstance(comp, list) else comp
        lang, code = extract_code(content)

        if not code or not ins:
            rewards.append(-0.5)
            continue

        score = 0
        total = len(ins)

        for i, o in zip(ins, outs):
            out, status = run_code(code, lang, i)
            if status == "ok" and out == o.strip():
                score += 1
                
        rewards.append(score / total if total > 0 else 0.0)

    return rewards

# ==========================================
# 4. TRAINING & LOGGING CALLBACK
# ==========================================
class EpochSaveCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        save_path = os.path.join(args.output_dir, "latest_epoch_weights")
        print(f"\n[Epoch {state.epoch}] Saving overwritten checkpoint to {save_path}")
        kwargs["model"].save_pretrained(save_path)
        kwargs["tokenizer"].save_pretrained(save_path)
    

training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, 
    num_generations = 4,
    
    # max_prompt_length = 1024,
    max_completion_length = 1024,
    num_train_epochs = 1,
    save_strategy="no",
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "outputs",
    importance_sampling_level = "sequence",
    loss_type = "dr_grpo",
    # save_strategy="steps",   # or "epoch"
    save_steps=2,          # save every 1 steps
    save_total_limit=2,      # keep last 3 checkpoints
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
    callbacks=[EpochSaveCallback()]
)

print("Starting GRPO Training...")
trainer.train()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


KeyboardInterrupt: 

In [ ]:

# ==========================================
# 5. POST-TRAINING VERIFICATION
# ==========================================
print("\n--- Running Manual Verification ---")
val_sample = processed_dataset[-1]

# BASE MODEL CHANGE: Use direct text tokenization instead of apply_chat_template
prompt_text = val_sample["prompt"]
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

print(f"Generating solution for Problem: {val_sample['id']}")
outputs = model.generate(
    **inputs, 
    max_new_tokens=1024, 
    pad_token_id=tokenizer.pad_token_id
)

# Decode only the generated portion
generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("\n--- Generated Output ---")
print(generated_text)
print("------------------------")

format_score = format_reward_func([generated_text])[0]
correctness_score = correctness_reward_func(
    [generated_text], 
    [val_sample["input_tests"]], 
    [val_sample["output_tests"]]
)[0]

print("\n--- Metrics ---")
print(f"Format Reward  : {format_score}")
print(f"Correctness    : {correctness_score} (Percentage of Test Cases Passed)")